In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras import layers, models, optimizers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

PROJECT_ROOT = "/content/drive/MyDrive/flower-multilabel-project"
RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")

with open(os.path.join(PROJECT_ROOT, "data", "metadata", "classes.json"), encoding="utf-8") as f:
    CLASSES = json.load(f)                      # giữ đúng thứ tự cố định (CON-05)
CLASS_IDS = [c["id"] for c in CLASSES]
NUM_CLASSES = len(CLASS_IDS)
IMG_SIZE = 224

Mounted at /content/drive


In [3]:
filepaths, labels = [], []
for idx, cls_id in enumerate(CLASS_IDS):
    folder = os.path.join(RAW_DIR, cls_id)
    for fname in os.listdir(folder):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            filepaths.append(os.path.join(folder, fname))
            labels.append(idx)

print(f"Tổng số ảnh: {len(filepaths)}")
for idx, cls_id in enumerate(CLASS_IDS):
    print(f"  {cls_id}: {labels.count(idx)} ảnh")

Tổng số ảnh: 5010
  hoa_hong: 782 ảnh
  hoa_huong_duong: 733 ảnh
  hoa_cuc: 764 ảnh
  hoa_tulip: 984 ảnh
  hoa_bo_cong_anh: 1052 ảnh
  hoa_sen: 137 ảnh
  hoa_lan: 100 ảnh
  hoa_mai: 159 ảnh
  hoa_dao: 167 ảnh
  hoa_oai_huong: 132 ảnh


In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    filepaths, labels, test_size=0.30, stratify=labels, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 3507 | Val: 751 | Test: 752


In [5]:
def load_and_preprocess(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.15)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

def make_dataset(paths, labels, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=42)
    ds = ds.map(lambda p, l: load_and_preprocess(p, l, augment),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train, y_train, augment=True, shuffle=True)
val_ds   = make_dataset(X_val, y_val)
test_ds  = make_dataset(X_test, y_test)

In [6]:
class_weights = compute_class_weight(
    class_weight="balanced", classes=np.arange(NUM_CLASSES), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)   # lớp Lan sẽ có trọng số cao hơn hẳn

{0: np.float64(0.6411334552102377), 1: np.float64(0.683625730994152), 2: np.float64(0.6555140186915888), 3: np.float64(0.5089985486211901), 4: np.float64(0.4764945652173913), 5: np.float64(3.653125), 6: np.float64(5.01), 7: np.float64(3.1594594594594594), 8: np.float64(2.9974358974358974), 9: np.float64(3.770967741935484)}


In [7]:
base_model = DenseNet121(include_top=False, weights="imagenet",
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)   # softmax, khác model đa nhãn

model = models.Model(inputs, outputs, name="densenet121_singlelabel")
model.compile(optimizer=optimizers.Adam(1e-4),
              loss="categorical_crossentropy", metrics=["accuracy"])

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
]
history1 = model.fit(train_ds, validation_data=val_ds, epochs=20,
                      class_weight=class_weight_dict, callbacks=callbacks)

Epoch 1/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 894s 8s/step - accuracy: 0.3191 - loss: 1.9502 - val_accuracy: 0.6897 - val_loss: 1.2969
Epoch 2/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 20s 178ms/step - accuracy: 0.6225 - loss: 1.0349 - val_accuracy: 0.8096 - val_loss: 0.8097
Epoch 3/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 19s 167ms/step - accuracy: 0.7328 - loss: 0.6791 - val_accuracy: 0.8455 - val_loss: 0.5987
Epoch 4/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 18s 159ms/step - accuracy: 0.7913 - loss: 0.5152 - val_accuracy: 0.8615 - val_loss: 0.5219
Epoch 5/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 18s 160ms/step - accuracy: 0.8186 - loss: 0.4215 - val_accuracy: 0.8748 - val_loss: 0.4406
Epoch 6/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 18s 159ms/step - accuracy: 0.8392 - loss: 0.3686 - val_accuracy: 0.8815 - val_loss: 0.4019
Epoch 7/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 21s 164ms/step - accuracy: 0.8537 - loss: 0.3294 - val_accuracy: 0.8828 - val_loss: 0.3722
Epoch 8/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 19s 170ms/step - accuracy: 0.8677 - loss: 0.2

In [10]:
base_model.trainable = True
fine_tune_at = int(len(base_model.layers) * 0.7)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=optimizers.Adam(1e-5),
              loss="categorical_crossentropy", metrics=["accuracy"])
callbacks2 = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
history2 = model.fit(train_ds, validation_data=val_ds, epochs=15,
                      class_weight=class_weight_dict, callbacks=callbacks2)

Epoch 1/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 152s 754ms/step - accuracy: 0.8794 - loss: 0.3089 - val_accuracy: 0.9134 - val_loss: 0.2676
Epoch 2/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 18s 160ms/step - accuracy: 0.9005 - loss: 0.1984 - val_accuracy: 0.9095 - val_loss: 0.2711
Epoch 3/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 21s 166ms/step - accuracy: 0.9196 - loss: 0.1696 - val_accuracy: 0.9081 - val_loss: 0.2600
Epoch 4/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 20s 160ms/step - accuracy: 0.9170 - loss: 0.1552 - val_accuracy: 0.9201 - val_loss: 0.2431
Epoch 5/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 18s 163ms/step - accuracy: 0.9287 - loss: 0.1351 - val_accuracy: 0.9268 - val_loss: 0.2324
Epoch 6/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 21s 169ms/step - accuracy: 0.9430 - loss: 0.1153 - val_accuracy: 0.9268 - val_loss: 0.2231
Epoch 7/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 19s 161ms/step - accuracy: 0.9430 - loss: 0.1038 - val_accuracy: 0.9254 - val_loss: 0.2129
Epoch 8/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 18s 162ms/step - accuracy: 0.9498 - loss: 

In [11]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")

# Per-class accuracy để kiểm tra riêng lớp Lan có ổn không
y_true, y_pred = [], []
for imgs, lbls in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(lbls.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=CLASS_IDS))

model.save(os.path.join(PROJECT_ROOT, "models", "final", "flower_singlelabel_model.keras"))

24/24 ━━━━━━━━━━━━━━━━━━━━ 136s 6s/step - accuracy: 0.9362 - loss: 0.1992
Test accuracy: 0.9362
                 precision    recall  f1-score   support

       hoa_hong       0.90      0.91      0.90       118
hoa_huong_duong       0.92      0.99      0.96       110
        hoa_cuc       0.96      0.95      0.96       115
      hoa_tulip       0.91      0.90      0.90       148
hoa_bo_cong_anh       0.97      0.92      0.94       158
        hoa_sen       1.00      1.00      1.00        20
        hoa_lan       0.88      1.00      0.94        15
        hoa_mai       0.96      0.92      0.94        24
        hoa_dao       1.00      1.00      1.00        25
  hoa_oai_huong       0.90      1.00      0.95        19

       accuracy                           0.94       752
      macro avg       0.94      0.96      0.95       752
   weighted avg       0.94      0.94      0.94       752

